# BMCLab original C3D gait dataset

Real optical motion capture of overground walking in Parkinson's disease. The original archive contains C3D and processed gait files, while PDGinfo.xlsx supplies participant and ON/OFF metadata. The prepared private Kaggle input minimizes this to two selected subjects and four clinical columns; the full spreadsheet stays local. The present animations use anatomical C3D markers, never the CARE-PD SMPL derivative. The Figshare source is CC BY 4.0.

**Scope.** Exploratory analysis of authentic recordings only. No disease classifier is trained. The selected animations show measured marker locations or estimated keypoints from source videos; a pose animation is not the original video.

**Source:** https://doi.org/10.6084/m9.figshare.14896881 ; https://doi.org/10.3389/fnins.2023.992585

## 1. Environment and reproducibility

The following functions validate the actual file layout, coordinates, frame IDs, and marker names. Kaggle inputs are read from `/kaggle/input`; figures and videos are written to `/kaggle/working/outputs`. On a local checkout, the same cells read `raw/` and write `outputs/`.

In [ ]:
"""Measured pose loading, quality checks, and traceable animation rendering."""
from __future__ import annotations

import csv
import json
import re
from xml.etree import ElementTree as ET
from zipfile import ZipFile
from dataclasses import dataclass
from pathlib import Path

import numpy as np

HALPE_NAMES = ["nose", "left_eye", "right_eye", "left_ear", "right_ear",
 "left_shoulder", "right_shoulder", "left_elbow", "right_elbow", "left_wrist", "right_wrist",
 "left_hip", "right_hip", "left_knee", "right_knee", "left_ankle", "right_ankle",
 "head", "neck", "hip", "left_big_toe", "right_big_toe", "left_small_toe", "right_small_toe",
 "left_heel", "right_heel"]
HALPE_EDGES = [(17,18),(18,5),(18,6),(5,7),(7,9),(6,8),(8,10),(18,19),
 (19,11),(19,12),(11,13),(13,15),(12,14),(14,16),(15,20),(15,22),(15,24),
 (16,21),(16,23),(16,25)]
CPM_NAMES = ["head","neck","Lsho","Lelb","Lwri","Rsho","Relb","Rwri",
 "Lhip","Lkne","Lank","Rhip","Rkne","Rank"]
CPM_EDGES = [(0,1),(1,2),(2,3),(3,4),(1,5),(5,6),(6,7),(1,8),(8,9),(9,10),
 (1,11),(11,12),(12,13),(8,11)]
BM_MARKERS = ["CLAV","C7","RSHO","REL","RWL","LSHO","LEL","LWL",
 "R.ASIS","L.ASIS","R.Knee","R.Ankle","R.Heel","R.MT2",
 "L.Knee","L.Ankle","L.Heel","L.MT2"]
BM_EDGES = [(1,0),(0,2),(2,3),(3,4),(0,5),(5,6),(6,7),
 (0,8),(0,9),(8,9),(8,10),(10,11),(11,12),(11,13),
 (9,14),(14,15),(15,16),(15,17)]


@dataclass
class Motion:
    dataset: str
    source: str
    points: np.ndarray       # frames x joints x dimension; metres for BMCLab, pixels for 2D
    confidence: np.ndarray   # frames x joints
    frame_ids: np.ndarray
    fps: float | None
    names: list[str]
    edges: list[tuple[int,int]]
    label: str
    subject: str | None = None
    warning: str = ""
    clinical_label: str | None = None


def _numeric_frame(value):
    m = re.search(r"(\d+)(?:\.[A-Za-z]+)?$", str(value))
    if not m:
        raise ValueError(f"No numeric frame ID: {value}")
    return int(m.group(1))


def load_mmu(path: str | Path, confidence_threshold=0.25) -> Motion:
    path = Path(path)
    rows = json.loads(path.read_text(encoding="utf-8"))
    by_frame = {}
    for row in rows:
        arr = np.asarray(row["keypoints"], dtype=float)
        if arr.size != 408:
            continue
        frame = _numeric_frame(row["image_id"])
        by_frame.setdefault(frame, []).append((row, arr.reshape(136, 3)))
    if len(by_frame) < 2:
        raise ValueError(f"No reconstructible sequence: {path}")
    selected, previous = [], None
    for frame in sorted(by_frame):
        candidates = by_frame[frame]
        def cost(item):
            row, pose = item
            center = np.nanmean(pose[[11,12], :2], axis=0)
            score = float(np.asarray(row.get("score", 0)).reshape(-1)[0])
            if previous is None:
                return -score
            return np.linalg.norm(center-previous) - min(score, 10.0) * 5
        chosen = min(candidates, key=cost)
        previous = np.nanmean(chosen[1][[11,12], :2], axis=0)
        selected.append((frame, chosen[1]))
    frame_ids = np.array([x[0] for x in selected], dtype=int)
    pose = np.stack([x[1] for x in selected])
    points = pose[:, :26, :2]
    conf = pose[:, :26, 2]
    points[conf < confidence_threshold] = np.nan
    label = next((part for part in path.parts if part.upper() in {"MILD","MODERATE","NORMAL","SEVERE"}),
                 path.stem.split("_")[0]).title()
    duplicate_count = sum(len(v)-1 for v in by_frame.values())
    gaps = int(np.sum(np.diff(frame_ids) > 1))
    return Motion("MMU",path.name,points,conf,frame_ids,None,HALPE_NAMES,HALPE_EDGES,label,
                  warning=f"{duplicate_count} extra detections; {gaps} frame gaps; source FPS unavailable",
                  clinical_label=label)


def load_bmclab(path: str | Path) -> Motion:
    import ezc3d
    path = Path(path)
    c = ezc3d.c3d(str(path))
    params = c["parameters"]["POINT"]
    labels = list(params["LABELS"]["value"])
    units = params["UNITS"]["value"][0].strip().lower()
    if units not in {"mm","m"}:
        raise ValueError(f"Unexpected C3D point units: {units}")
    raw = c["data"]["points"]
    if not all(name in labels for name in BM_MARKERS):
        raise ValueError(f"Missing anatomical markers: {set(BM_MARKERS)-set(labels)}")
    points = np.transpose(raw[:3, [labels.index(n) for n in BM_MARKERS], :], (2,1,0)).copy()
    if units == "mm":
        points /= 1000.0
    residual = np.transpose(raw[3, [labels.index(n) for n in BM_MARKERS], :], (1,0))
    valid = np.isfinite(points).all(axis=2) & np.isfinite(residual) & (residual >= 0)
    points[~valid] = np.nan
    fps = float(params["RATE"]["value"][0])
    sub = re.search(r"SUB\d+",path.stem,re.I)
    med = re.search(r"_(on|off)_",path.stem,re.I)
    return Motion("BMCLab",path.name,points,valid.astype(float),np.arange(len(points)),fps,
                  BM_MARKERS,BM_EDGES,(med.group(1).upper() if med else "Unknown"),
                  subject=(sub.group(0) if sub else None),
                  warning=f"Original C3D coordinates; {units} converted to metres; C3D Y is vertical",
                  clinical_label=(med.group(1).upper() if med else None))


def load_pose_file(path: str | Path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def load_pose_record(path: str | Path, key: str, subjects: dict | None = None) -> Motion:
    path = Path(path)
    record = load_pose_file(path)[key]
    if "idx" in record:
        raise ValueError("Leg-agility split indices require a separate mapping; select communication or drinking")
    lens = [len(record["position"][name]) for name in CPM_NAMES]
    n = min(lens)
    points = np.stack([np.asarray(record["position"][name][:n],float) for name in CPM_NAMES],axis=1)
    confidence = np.stack([np.asarray(record["resp"][name][:n],float) for name in CPM_NAMES],axis=1)
    if points.shape != (n,14,2):
        raise ValueError("Unexpected CPM trajectory shape")
    points[~np.isfinite(points).all(axis=2)] = np.nan
    points[confidence < 0.2] = np.nan
    task = "Communication" if path.name.startswith("Communication") else "Drinking"
    trial = re.split(r"[-\s]",key)[0]
    ratings_path=path.parent/"UDysRS.txt"
    ratings=load_pose_file(ratings_path).get(task,{}).get(trial) if ratings_path.exists() else None
    return Motion("Pose estimation",f"{path.name}:{key}",points,confidence,np.arange(n),30.,
                  CPM_NAMES,CPM_EDGES,task,subject=(subjects or {}).get(trial),
                  warning="2D camera coordinates; 30 FPS from Li et al. 2018; no original video in archive",
                  clinical_label=("UDysRS subscores " + json.dumps(ratings) if ratings is not None else None))


def quality(m: Motion) -> dict:
    p=m.points
    valid=np.isfinite(p).all(axis=2)
    disp=np.linalg.norm(np.diff(p,axis=0),axis=2)
    return {"dataset":m.dataset,"source":m.source,"label":m.label,"subject":m.subject or "Unknown",
            "frames":len(p),"duration_s":(m.frame_ids[-1]-m.frame_ids[0]+1)/m.fps if m.fps else None,
            "source_fps":m.fps,"joint_valid_fraction":float(valid.mean()),
            "median_displacement_per_frame":float(np.nanmedian(disp)),
            "frame_gaps":int(np.sum(np.diff(m.frame_ids)>1)),"warning":m.warning}


def _view_indices(m: Motion, out_fps=20, max_seconds=8):
    if m.fps:
        end=min(float(m.frame_ids[-1]-m.frame_ids[0])/m.fps,max_seconds)
        times=np.arange(0,end+1e-9,1/out_fps)
        ids=np.searchsorted(m.frame_ids, m.frame_ids[0]+times*m.fps).clip(0,len(m.frame_ids)-1)
    else:
        # Source FPS absent: one recorded pose per output frame; speed is illustrative only.
        ids=np.arange(min(len(m.frame_ids),int(max_seconds*out_fps)))
    return np.unique(ids)


def render(m: Motion, video: str | Path, preview: str | Path, out_fps=20, max_seconds=8):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation, FFMpegWriter
    import shutil
    video,preview=Path(video),Path(preview)
    video.parent.mkdir(parents=True,exist_ok=True); preview.parent.mkdir(parents=True,exist_ok=True)
    encoder=shutil.which("ffmpeg")
    if encoder is None:
        import imageio_ffmpeg
        encoder=imageio_ffmpeg.get_ffmpeg_exe()
    matplotlib.rcParams["animation.ffmpeg_path"] = encoder
    idx=_view_indices(m,out_fps,max_seconds)
    if len(idx)<2: raise ValueError("Fewer than two animation frames")
    # In these files C3D Y is vertical: shoulder Y exceeds hip, knee, and heel Y.
    # Matplotlib uses its third displayed axis as vertical, so display X/Z/Y.
    pts=m.points[idx][:,:,[0,2,1]] if m.points.shape[2]==3 else m.points[idx]
    good=pts[np.isfinite(pts).all(axis=2)]
    lo=np.nanpercentile(good,1,axis=0); hi=np.nanpercentile(good,99,axis=0)
    span=np.maximum(hi-lo,0.1 if pts.shape[2]==3 else 10)
    lo-=span*0.12; hi+=span*0.12
    if pts.shape[2]==3:
        fig=plt.figure(figsize=(6,6)); ax=fig.add_subplot(111,projection="3d")
        # Keep equal physical scale and include the ground plane.
        center=(lo+hi)/2; radius=max((hi-lo))/2
        limits=[(float(c-radius),float(c+radius)) for c in center]
        def setup():
            ax.set_xlim(*limits[0]); ax.set_ylim(*limits[1]); ax.set_zlim(*limits[2]);
            ax.set_xlabel("X (m)"); ax.set_ylabel("Z (m)"); ax.set_zlabel("Y up (m)")
            ax.view_init(elev=16,azim=-65)
            ax.set_box_aspect((1,1,1),zoom=1.2)
            xx=np.linspace(*limits[0],2); yy=np.linspace(*limits[1],2)
            X,Y=np.meshgrid(xx,yy); ax.plot_surface(X,Y,np.zeros_like(X),alpha=.08,color="gray")
    else:
        fig,ax=plt.subplots(figsize=(6,6))
        limits=[(float(lo[i]),float(hi[i])) for i in range(2)]
        def setup():
            ax.set_xlim(*limits[0]); ax.set_ylim(*limits[1]); ax.invert_yaxis(); ax.set_aspect("equal",adjustable="box")
            ax.set_xlabel("image X (pixels)"); ax.set_ylabel("image Y (pixels)")
    def draw(j):
        ax.clear(); setup(); p=pts[j]
        for a,b in m.edges:
            if np.isfinite(p[[a,b]]).all():
                q=p[[a,b]]
                if p.shape[1]==3: ax.plot(q[:,0],q[:,1],q[:,2],color="#2563eb",lw=2.4)
                else: ax.plot(q[:,0],q[:,1],color="#2563eb",lw=2.4)
        ok=np.isfinite(p).all(axis=1)
        if p.shape[1]==3: ax.scatter(p[ok,0],p[ok,1],p[ok,2],s=13,color="#ef4444",depthshade=False)
        else:
            sizes=10+22*np.clip(m.confidence[idx[j],ok],0,1)
            ax.scatter(p[ok,0],p[ok,1],s=sizes,color="#ef4444")
        ax.set_title(f"{m.dataset} | {m.source}\n{m.label} | source frame {m.frame_ids[idx[j]]}",fontsize=10)
    draw(len(idx)//2); fig.savefig(preview,dpi=130,bbox_inches="tight")
    anim=FuncAnimation(fig,draw,frames=len(idx),interval=1000/out_fps)
    anim.save(str(video),writer=FFMpegWriter(fps=out_fps,codec="libx264",bitrate=1200))
    plt.close(fig)
    return {"dataset":m.dataset,"source_recording_id":m.source,"selected_sequence_range":f"{m.frame_ids[idx[0]]}-{m.frame_ids[idx[-1]]}",
            "original_fps":m.fps if m.fps else "Unknown","output_fps":out_fps,"output_filename":str(video),
            "rendered_frames":len(idx),"modality":"3D" if m.points.shape[2]==3 else "2D",
            "status":"measured markers" if m.dataset=="BMCLab" else "estimated 2D pose from real video",
            "clinical_label":m.clinical_label or "Unavailable","subject_id":m.subject or "Unknown","preview_filename":str(preview)}


def save_manifest(rows, path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    with path.open("w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)


def read_simple_xlsx(path):
    """Read the first worksheet of PDGinfo.xlsx without an optional Excel package."""
    ns={"m":"http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    with ZipFile(path) as z:
        strings=["".join(x.itertext()) for x in ET.fromstring(z.read("xl/sharedStrings.xml")).findall("m:si",ns)]
        root=ET.fromstring(z.read("xl/worksheets/sheet1.xml"))
    rows=[]
    for row in root.findall(".//m:sheetData/m:row",ns):
        cells={}
        for c in row.findall("m:c",ns):
            value=c.find("m:v",ns)
            if value is None: continue
            col=re.match(r"[A-Z]+",c.get("r")).group()
            cells[col]=strings[int(value.text)] if c.get("t")=="s" else value.text
        rows.append(cells)
    header=rows[0]
    return [{header[k]:v for k,v in row.items() if k in header} for row in rows[1:]]


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown
ROOT=Path('/kaggle/working') if Path('/kaggle/input').exists() else Path.cwd()
OUTPUT=ROOT/'outputs'; (OUTPUT/'figures').mkdir(parents=True,exist_ok=True); (OUTPUT/'tables').mkdir(parents=True,exist_ok=True)
import os
import importlib.util
if importlib.util.find_spec('ezc3d') is None:
    import subprocess, sys
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','ezc3d'])
DATASET='bmclab'
INPUT=Path(os.environ.get('PARKINSON_BM_INPUT','/kaggle/input' if Path('/kaggle/input').exists() else 'raw/bmclab'))
all_files=sorted(INPUT.rglob('*.c3d'))
assert all_files,'C3D input missing'
selected=['SUB01_off_walk_14.c3d','SUB01_on_walk_19.c3d','SUB02_off_walk_6.c3d']
motions=[load_bmclab(next(p for p in all_files if p.name==name)) for name in selected]
inv_file=next(iter(INPUT.rglob('bmclab_inventory.csv')),Path('outputs/tables/bmclab_inventory.csv'))
meta_file=next(iter(INPUT.rglob('bmclab_selected_metadata.csv')),Path('raw/bmclab/PDGinfo.xlsx'))

## 2. Data loading and structure

Inspect actual source records before interpreting summary counts. The loaded `Motion` objects retain source IDs and frame indices so every figure and video can be traced back to a recording.

In [ ]:
overview=pd.DataFrame([quality(m) for m in motions]); display(overview)
print('Example source:', motions[0].source)
print('Shape (frames, joints, coordinates):', motions[0].points.shape)
print('First joint names:', motions[0].names[:20])
print('First frame IDs:', motions[0].frame_ids[:10].tolist())

## 3. Dataset statistics

Counts from source inventories and annotations are distinguished from the three selected trajectories. A recording is a temporal sequence, not a single pose frame. Unknown capture rates remain unknown; output video FPS does not supply a missing source FPS.

In [ ]:
inventory=pd.read_csv(inv_file);meta=pd.read_csv(meta_file) if meta_file.suffix=='.csv' else pd.DataFrame(read_simple_xlsx(meta_file));print('Inventory C3D files:',len(inventory),'subjects:',inventory.subject.nunique())
display(inventory.groupby(['trial_type','medication']).size().rename('C3D files').reset_index())
fig,ax=plt.subplots(figsize=(6,3));inventory.query("trial_type=='walk'").groupby('medication').size().plot.bar(ax=ax,color='#2563eb');ax.set(title='Original archive walking trials',ylabel='C3D files');plt.show()
print('Metadata participant rows available in this environment:',meta.ID.nunique(),'| selected C3D frames:',sum(len(m.points) for m in motions))
print('Selected source rates and units:',[(m.source,m.fps,m.warning) for m in motions])

## 3a. Distribution shape and dataset skewness

The reference EDA compares recording durations and label distributions. Here, numeric skewness uses the adjusted Fisher–Pearson sample coefficient (`pandas.Series.skew`): positive values have a longer right tail and negative values a longer left tail. It is undefined for fewer than three observations or a constant series. Class and task names are categorical, so their **imbalance** is reported as counts, shares, and largest-to-smallest count ratios instead of assigning numeric codes and calculating skewness. Tables distinguish the full accessible inventory from the three selected animations.

In [ ]:
"""BMCLab distribution profile; embedded in the generated notebook."""
from IPython.display import display, Markdown

subject_counts = inventory.groupby('subject').size().rename('C3D trials')
medication = inventory.medication.value_counts().reindex(['ON','OFF'], fill_value=0)
trial_types = inventory.trial_type.value_counts()
selected_durations = pd.Series(overview.duration_s.dropna().to_numpy(), dtype=float)
on_scores = pd.to_numeric(meta['ON - UPDRS-III'], errors='coerce').dropna()
off_scores = pd.to_numeric(meta['OFF - UPDRS-III'], errors='coerce').dropna()
display(pd.DataFrame([{'C3D_recordings':len(inventory), 'participants':subject_counts.size,
                       'source_FPS':'150 Hz in the three selected headers',
                       'pose_dimension':'measured 3D metres',
                       'labels':'ON/OFF; FoG group and UPDRS-III at participant level'}]))

# The full local spreadsheet contains the actual 0–4 walking item. The
# approved two-subject Kaggle input contains totals only, so skip there.
walking_columns={'ON':'ON - UPDRS-III - walking','OFF':'OFF - UPDRS-III - walking'}
if all(col in meta.columns for col in walking_columns.values()):
    walking_counts=pd.DataFrame({'score':range(5)})
    for medication_label,column in walking_columns.items():
        values=pd.to_numeric(meta[column],errors='coerce').dropna()
        assert values.between(0,4).all() and values.eq(values.round()).all(), column
        walking_counts[medication_label]=values.astype(int).value_counts().reindex(range(5),fill_value=0).values
    display(walking_counts)
    walking_counts.to_csv(OUTPUT/'tables'/'bmclab_score_categories.csv',index=False)
    fig,ax=plt.subplots(figsize=(7,4))
    x=np.arange(5);width=.38
    ax.bar(x-width/2,walking_counts.ON,width,label='ON',color='#0f766e')
    ax.bar(x+width/2,walking_counts.OFF,width,label='OFF',color='#ea580c')
    ax.set_xticks(x,[str(i) for i in x])
    ax.set(xlabel='Recorded UPDRS-III walking item score',ylabel='Participants',
           title='BMCLab UPDRS gait item by medication')
    ax.legend();fig.tight_layout()
    fig.savefig(OUTPUT/'figures'/'bmclab_score_categories.png',dpi=150);plt.show()
    display(Markdown('Scores **1–4** are shown as recorded; **0** is also shown because it occurs in the source. Counts are participant ratings, not numbers of C3D trials.'))
else:
    display(Markdown('The approved private Kaggle sample contains UPDRS-III **totals** but not the 0–4 walking-item columns. Its gait-category bar chart is available only in the full local-data execution; totals cannot be relabeled as 1–4 gait scores.'))

def numeric_profile(name, values, scope, units):
    values = pd.Series(values, dtype=float).dropna()
    skew = values.skew() if len(values) >= 3 and values.nunique() > 1 else float('nan')
    return {'measure':name, 'scope':scope, 'n':len(values), 'units':units,
            'median':values.median() if len(values) else float('nan'),
            'p90':values.quantile(.9) if len(values) else float('nan'), 'skewness':skew}

profile = pd.DataFrame([
    numeric_profile('C3D trials per participant', subject_counts, 'full 935-file archive inventory', 'trials'),
    numeric_profile('selected walk duration', selected_durations, 'three selected C3Ds only', 'seconds'),
    numeric_profile('ON UPDRS-III', on_scores, f'{meta.ID.nunique()} metadata participants available here', 'score'),
    numeric_profile('OFF UPDRS-III', off_scores, f'{meta.ID.nunique()} metadata participants available here', 'score')
])
display(profile.round(2))
profile.to_csv(OUTPUT/'tables'/'bmclab_distribution_profile.csv', index=False)
display(pd.DataFrame({'medication':medication.index, 'C3D_files':medication.values,
                      'share_pct':(100*medication/medication.sum()).round(1).values}))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(subject_counts, bins=range(int(subject_counts.min()),int(subject_counts.max())+2),
             color='#2563eb', edgecolor='white')
axes[0].axvline(subject_counts.median(), color='#dc2626', linestyle='--', label=f'Median {subject_counts.median():.0f}')
axes[0].set(xlabel='C3D trials per participant', ylabel='Participants', title='Full archive trial-count distribution')
axes[0].legend()
axes[1].bar(medication.index, medication.values, color=['#0f766e','#ea580c'])
axes[1].set(ylabel='C3D files', title='Archive medication conditions')
axes[2].bar(range(len(overview)), overview.duration_s, color='#7c3aed')
axes[2].set_xticks(range(len(motions)),
                   [f"{m.subject} {m.label} #{m.source.split('_')[-1].split('.')[0]}" for m in motions],
                   rotation=20, ha='right')
axes[2].set(ylabel='Seconds at measured 150 Hz', title='Selected walking durations only')
fig.tight_layout(); fig.savefig(OUTPUT/'figures'/'bmclab_distribution_skewness.png', dpi=150); plt.show()

trial_skew = profile.loc[profile.measure=='C3D trials per participant','skewness'].iloc[0]
med_ratio = medication.max()/medication[medication>0].min()
score_scope = 'Full local metadata' if meta.ID.nunique()>2 else 'Private Kaggle metadata for two selected participants'
display(Markdown(f"**Interpretation.** The full inventory has {len(inventory)} C3Ds from {subject_counts.size} participants, including {trial_types.get('walk',0)} walks and {trial_types.get('static',0)} static trials. Per-participant trial-count skewness is **{trial_skew:.2f}**; this measures how recording volume differs by participant, not gait severity. ON/OFF file counts have a {med_ratio:.2f}:1 imbalance ratio, but repeated trials from the same person are not independent participants. {score_scope} supplies {meta.ID.nunique()} participant rows; score skewness is left undefined when fewer than three usable scores are present. Only the three selected C3Ds have measured durations in this notebook."))


## 4. Data quality

Valid joints have finite coordinates and pass the available confidence or C3D residual checks. Frame gaps and low-confidence detections can distort apparent velocity. The following chart describes only the selected recordings; it is not a clinical quality score.

In [ ]:
q=overview[['source','joint_valid_fraction','frame_gaps','median_displacement_per_frame']].copy(); display(q)
fig,ax=plt.subplots(figsize=(9,3)); ax.bar(range(len(q)),q.joint_valid_fraction,color='#2563eb'); ax.set_xticks(range(len(q)),[str(x)[:25] for x in q.source],rotation=25,ha='right'); ax.set_ylim(0,1); ax.set_ylabel('Valid joint fraction'); ax.set_title('Pose availability in selected recordings'); fig.tight_layout(); fig.savefig(OUTPUT/'figures'/f'{DATASET}_quality.png',dpi=150); plt.show()
print('Reconstructible selected recordings:',int((overview.joint_valid_fraction>0.5).sum()),'of',len(overview))

### Missing joints, confidence, and abrupt changes

The joint chart identifies locations that are repeatedly unavailable. For 2D data the confidence histogram comes from AlphaPose or CPM, and its scale is specific to that estimator. The displacement tail is a tracking-review flag, not proof of an outlier symptom.

In [ ]:
m=motions[0];valid=np.isfinite(m.points).all(axis=2);missing=1-valid.mean(axis=0)
fig,axes=plt.subplots(1,2,figsize=(12,3.5)); axes[0].bar(range(len(m.names)),missing,color='#ef4444');axes[0].set_xticks(range(len(m.names)),m.names,rotation=90,fontsize=7);axes[0].set(ylabel='Missing fraction',title='Selected source joint availability')
if m.dataset=='BMCLab':
    d=np.linalg.norm(np.diff(m.points,axis=0),axis=2);axes[1].hist(d[np.isfinite(d)].ravel(),bins=40,color='#2563eb');axes[1].set(xlabel='Metres per source frame',title='Marker displacement distribution')
else:
    c=m.confidence[np.isfinite(m.confidence)];axes[1].hist(c,bins=40,color='#2563eb');axes[1].set(xlabel='Estimator response',title='Keypoint confidence distribution')
fig.tight_layout();fig.savefig(OUTPUT/'figures'/f'{DATASET}_missing_confidence.png',dpi=150);plt.show()
for m in motions:
    d=np.linalg.norm(np.diff(m.points,axis=0),axis=2);print(m.source,'99th percentile per-frame joint displacement:',round(float(np.nanpercentile(d,99)),3),'frame gaps:',quality(m)['frame_gaps'])

## 5. Human movement characteristics

Joint trajectories and frame displacement reveal temporal variation. Two-dimensional values are camera pixels, affected by distance and viewpoint. BMCLab coordinates are converted from verified millimetres to metres; the anatomical marker positions show that C3D Y is vertical. These are descriptive kinematics, not symptom diagnoses.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,3.5))
for m in motions:
    j=0 if m.dataset=='BMCLab' else (9 if m.dataset=='MMU' else 4)
    t=(m.frame_ids-m.frame_ids[0])/(m.fps or 20)
    axes[0].plot(t,m.points[:,j,1],label=m.source[:22])
    d=np.linalg.norm(np.diff(m.points[:,j,:],axis=0),axis=1)
    axes[1].plot(t[1:],d,label=m.source[:22],alpha=.8)
axes[0].set(xlabel='Time (s; MMU uses illustrative 20 FPS)',ylabel='C3D Y (m) or image Y (px)',title='Selected point trajectory')
axes[1].set(xlabel='Time (s; MMU illustrative)',ylabel='Displacement per source frame',title='Frame displacement')
axes[0].legend(fontsize=7);fig.tight_layout();fig.savefig(OUTPUT/'figures'/f'{DATASET}_movement.png',dpi=150);plt.show()

### Left and right elbow motion

An approximate joint angle is computed from shoulder, elbow, and wrist points. It depends on the marker or 2D projection definition. These curves illustrate within-recording movement variability and side differences; no clinical diagnosis follows from them.

In [ ]:
def elbow_angle(m,side):
    triplets={'MMU':{'L':(5,7,9),'R':(6,8,10)},'BMCLab':{'L':(5,6,7),'R':(2,3,4)},'Pose estimation':{'L':(2,3,4),'R':(5,6,7)}}
    a,b,c=triplets[m.dataset][side];u=m.points[:,a]-m.points[:,b];v=m.points[:,c]-m.points[:,b]
    dot=np.sum(u*v,axis=1);den=np.linalg.norm(u,axis=1)*np.linalg.norm(v,axis=1)
    return np.degrees(np.arccos(np.clip(dot/den,-1,1)))
m=motions[0];t=(m.frame_ids-m.frame_ids[0])/(m.fps or 20);fig,ax=plt.subplots(figsize=(9,3))
for side,color in [('L','#2563eb'),('R','#ef4444')]:ax.plot(t,elbow_angle(m,side),label=side,color=color,alpha=.8)
ax.set(xlabel='Time (s; MMU illustrative)',ylabel='Projected or spatial angle (degrees)',title='Selected recording elbow angles');ax.legend();fig.tight_layout();fig.savefig(OUTPUT/'figures'/f'{DATASET}_elbow_angles.png',dpi=150);plt.show()
print('Left/right median elbow angles:',{side:float(np.nanmedian(elbow_angle(m,side))) for side in ['L','R']})

### Pelvis movement rate and variability

The midpoint of the two hip or pelvic markers approximates body translation. First and second finite differences summarize movement rate and rate variability. For C3D these use metres per second and metres per second squared; CPM values are camera pixels, and MMU values remain pixels per frame because source FPS is unknown.

In [ ]:
pelvis_indices={'MMU':(11,12),'BMCLab':(8,9),'Pose estimation':(8,11)}
movement=[]
for m in motions:
    center=np.nanmean(m.points[:,pelvis_indices[m.dataset],:],axis=1)
    step=np.diff(m.frame_ids).astype(float);dt=step/m.fps if m.fps else step
    velocity=np.linalg.norm(np.diff(center,axis=0),axis=1)/dt
    acceleration=np.abs(np.diff(velocity))/dt[1:]
    movement.append({'source':m.source,'median_rate':float(np.nanmedian(velocity)),'rate_p95':float(np.nanpercentile(velocity,95)),'median_rate_change':float(np.nanmedian(acceleration)),'units':('m/s; m/s²' if m.dataset=='BMCLab' else 'px/s; px/s²' if m.fps else 'px/frame; px/frame²')})
movement=pd.DataFrame(movement);display(movement);movement.to_csv(OUTPUT/'tables'/f'{DATASET}_motion_rates.csv',index=False)
fig,ax=plt.subplots(figsize=(8,3));ax.bar(range(len(movement)),movement.median_rate,color='#2563eb');ax.set_xticks(range(len(movement)),[s[:22] for s in movement.source],rotation=20,ha='right');ax.set(ylabel='Median pelvis movement rate',title='Selected recording movement rates');fig.tight_layout();fig.savefig(OUTPUT/'figures'/f'{DATASET}_motion_rates.png',dpi=150);plt.show()

## 6. Clinical labels and movement

The available annotation belongs to the source trial or participant. Plots compare descriptive movement quantities only. A small selected sample cannot support clinical inference or ON/OFF treatment effect estimates.

In [ ]:
cols=['ID','FoG group','ON - UPDRS-III','OFF - UPDRS-III']
display(meta[cols].head(5));print('FoG groups:',meta['FoG group'].value_counts(dropna=False).to_dict())
fig,axes=plt.subplots(1,2,figsize=(9,3));meta['FoG group'].dropna().value_counts().plot.bar(ax=axes[0],color='#2563eb');axes[0].set(title='Participant FoG groups',ylabel='participants')
up=[pd.to_numeric(meta[c],errors='coerce').dropna() for c in cols[2:]];axes[1].boxplot(up,tick_labels=['ON','OFF']);axes[1].set(title='Available participant UPDRS-III',ylabel='score');fig.tight_layout();fig.savefig(OUTPUT/'figures'/f'{DATASET}_clinical.png',dpi=150);plt.show()
print('These participant-level ratings are not per-frame labels. ON/OFF comparisons require paired, subject-level analysis beyond three previews.')

## 7. Representative static skeletons

For each recording, the pose with the most visible joints is selected; ties use the frame nearest the midpoint. The plots use the dataset's own marker or keypoint definitions. Low-confidence MMU and CPM points are suppressed by the loader; missing segments remain absent. The 3D view maps source X/Z/Y to display X/Y/Z because source Y is vertical.

In [ ]:
fig=plt.figure(figsize=(13,4))
side_joints={'MMU':{'L wrist':9,'R wrist':10,'L ankle':15,'R ankle':16},'BMCLab':{'L wrist':7,'R wrist':4,'L ankle':15,'R ankle':11},'Pose estimation':{'L wrist':4,'R wrist':7,'L ankle':10,'R ankle':13}}
for i,m in enumerate(motions,1):
    visibility=np.isfinite(m.points).all(axis=2).sum(axis=1);best=np.flatnonzero(visibility==visibility.max());frame=best[np.argmin(np.abs(best-len(m.points)//2))];p=m.points[frame]
    if p.shape[1]==3:
        ax=fig.add_subplot(1,3,i,projection='3d'); v=p[:,[0,2,1]]
        for a,b in m.edges:
            if np.isfinite(v[[a,b]]).all(): ax.plot(v[[a,b],0],v[[a,b],1],v[[a,b],2],color='#2563eb')
        ax.scatter(v[:,0],v[:,1],v[:,2],s=8);ax.set(xlabel='X m',ylabel='Z m',zlabel='Y m')
        center=(np.nanmin(v,axis=0)+np.nanmax(v,axis=0))/2;radius=float(np.nanmax(np.nanmax(v,axis=0)-np.nanmin(v,axis=0)))*.6
        ax.set_xlim(center[0]-radius,center[0]+radius);ax.set_ylim(center[1]-radius,center[1]+radius);ax.set_zlim(center[2]-radius,center[2]+radius);ax.set_box_aspect((1,1,1))
        for label,j in side_joints[m.dataset].items():
            if np.isfinite(v[j]).all():ax.text(*v[j],label,fontsize=6)
    else:
        ax=fig.add_subplot(1,3,i)
        for a,b in m.edges:
            if np.isfinite(p[[a,b]]).all():ax.plot(p[[a,b],0],p[[a,b],1],color='#2563eb')
        ax.scatter(p[:,0],p[:,1],s=8);ax.invert_yaxis();ax.set_aspect('equal',adjustable='box')
        for label,j in side_joints[m.dataset].items():
            if np.isfinite(p[j]).all():ax.annotate(label,p[j],fontsize=6)
    ax.set_title(m.source[:25]+f'\nframe {m.frame_ids[frame]}',fontsize=8)
fig.tight_layout();fig.savefig(OUTPUT/'figures'/f'{DATASET}_static_skeletons.png',dpi=150);plt.show()

## 8. Animated measured or estimated movement

The selection is fixed by source filename and, for CPM, trial key. Each MP4 uses only poses from that recording. C3D playback is resampled from the 150 Hz source to 20 FPS without changing duration. CPM playback uses the 30 FPS capture rate reported by Li et al. MMU capture FPS is not included in its JSON, so 20 FPS playback is illustrative; its real duration is unknown. The manifest records the exact frame range and selected label.

In [ ]:
from IPython.display import display, Video
import csv
manifest=[];manifest_file=OUTPUT/'tables'/f'{DATASET}_simulation_manifest.csv'
previous=list(csv.DictReader(manifest_file.open(encoding='utf8'))) if manifest_file.exists() else []
for i,m in enumerate(motions,1):
    stem=f'{DATASET}_{i:02d}'
    video=OUTPUT/'videos'/DATASET/f'{stem}.mp4'; preview=OUTPUT/'figures'/DATASET/f'{stem}.png'
    cached=next((row for row in previous if row['source_recording_id']==m.source),None)
    row=cached if cached and video.exists() and preview.exists() else render(m,video,preview,out_fps=20,max_seconds=8)
    manifest.append(row)
    print(video, 'source:',m.source, 'frames:',row['rendered_frames'])
save_manifest(manifest,manifest_file)
display(pd.DataFrame(manifest))
display(Video(str(OUTPUT/'videos'/DATASET/f'{DATASET}_01.mp4'),embed=False))

## 9. Suitability for future VLM development

Measured 3D marker coordinates, 150 Hz timing and ON/OFF source naming make this the strongest basis for faithful 3D motion. The 18 displayed anatomical markers show limb and trunk movement; a mesh would require marker-model fitting, registration, and a separately licensed body model. Source videos are not included in the C3D archive.

An animation of sparse joints can support motion caption prototyping. A future VLM needs clear evidence about what images or source video are available and how clinical labels align with individual sequences. Pose coordinates alone do not provide texture, facial expression, or scene context.

## 10. Research summary

The table links each selected recording to its pose quality and rendered video. The distribution profile above gives the scope of each count or skewness value. Interpret duration only where original FPS is known. Review the quality chart, static skeletons, and videos together. No synthetic trajectory was substituted for a missing recording.

In [ ]:
summary=overview.copy(); summary['video']=[x['output_filename'] for x in manifest]; display(summary); summary.to_csv(OUTPUT/'tables'/f'{DATASET}_summary.csv',index=False)
print('Selected recordings:',len(summary),'| source frames:',int(summary.frames.sum()),'| mean valid-joint fraction:',round(float(summary.joint_valid_fraction.mean()),3))
print('Source FPS values:',sorted({str(x) for x in summary.source_fps}))
print('Distribution profile:',OUTPUT/'tables'/f'{DATASET}_distribution_profile.csv')